In [ ]:
# ===============================
# TRAIN BẰNG clean_data
# TEST BẰNG clean_test_data CÓ CHURN
# CHẠY 4 MODEL: Logistic Regression, Random Forest, Gradient Boosting, XGBoost
# ===============================

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier

print("=== BẮT ĐẦU TRAIN VÀ TEST MODEL DỰ ĐOÁN CHURN ===")

# ===============================
# 1. LOAD DATA
# ===============================

train_path = "/content/cleaned_data.csv"
test_path = "/content/cleaned_test_data.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Kích thước train:", train_df.shape)
print("Kích thước test:", test_df.shape)

# ===============================
# 2. KIỂM TRA CỘT CHURN
# ===============================

=== BẮT ĐẦU TRAIN VÀ TEST MODEL DỰ ĐOÁN CHURN ===
Kích thước train: (4504, 31)
Kích thước test: (1126, 31)
Số cột X_train: 29
Số cột X_test: 29
Cột train và test đã khớp hoàn toàn.
Số lượng Churn = 0 trong train: 3746
Số lượng Churn = 1 trong train: 758
scale_pos_weight: 4.941952506596306

Đang train model: Logistic Regression
Confusion Matrix:
[[731 205]
 [ 31 159]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.78      0.86       936
           1       0.44      0.84      0.57       190

    accuracy                           0.79      1126
   macro avg       0.70      0.81      0.72      1126
weighted avg       0.87      0.79      0.81      1126


Đang train model: Random Forest
Confusion Matrix:
[[935   1]
 [ 23 167]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       936
           1       0.99      0.88      0.93       190

    accuracy     

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
1,Random Forest,0.978686,0.994048,0.878947,0.932961,0.998828
3,XGBoost,0.926288,0.718367,0.926316,0.809195,0.977581
2,Gradient Boosting,0.920959,0.825806,0.673684,0.742029,0.941695
0,Logistic Regression,0.790409,0.436813,0.836842,0.574007,0.886083


Đã lưu bảng so sánh model tại: /content/model_comparison_results.csv

Model tốt nhất theo F1-score: Random Forest

Đã lưu file dự đoán model tốt nhất tại: /content/churn_prediction_best_model.csv


,CustomerID,Actual_Churn,Predicted_Churn,Churn_Probability
0,0,0,0,0.010000
1,0,0,0,0.023333
2,0,0,0,0.013333
3,0,0,0,0.010000
4,0,0,0,0.070000


In [ ]:

if "Churn" not in train_df.columns:
    raise ValueError("File cleaned_data.csv không có cột Churn.")

if "Churn" not in test_df.columns:
    raise ValueError("File cleaned_test_data.csv không có cột Churn.")

# ===============================
# 3. TÁCH X, y
# ===============================

X_train = train_df.drop(["Churn", "CustomerID"], axis=1, errors="ignore")
y_train = train_df["Churn"]

X_test = test_df.drop(["Churn", "CustomerID"], axis=1, errors="ignore")
y_test = test_df["Churn"]

# Lưu CustomerID để xuất kết quả
if "CustomerID" in test_df.columns:
    customer_id = test_df["CustomerID"]
else:
    customer_id = pd.Series(range(1, len(test_df) + 1), name="CustomerID")

# ===============================
# 4. ĐỒNG BỘ CỘT TRAIN VÀ TEST
# ===============================

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Số cột X_train:", X_train.shape[1])
print("Số cột X_test:", X_test.shape[1])

if list(X_train.columns) == list(X_test.columns):
    print("Cột train và test đã khớp hoàn toàn.")
else:
    print("Cột train và test chưa khớp.")

# ===============================
# 5. TÍNH scale_pos_weight CHO XGBOOST
# ===============================

negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

if positive_count == 0:
    scale_pos_weight = 1
else:
    scale_pos_weight = negative_count / positive_count

print("Số lượng Churn = 0 trong train:", negative_count)
print("Số lượng Churn = 1 trong train:", positive_count)
print("scale_pos_weight:", scale_pos_weight)

# ===============================
# 6. KHAI BÁO 4 MODEL
# ===============================

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        objective="binary:logistic",
        random_state=42,
        n_jobs=-1
    )
}

# ===============================
# 7. TRAIN VÀ TEST 4 MODEL
# ===============================

results = []
trained_models = {}

for model_name, model in models.items():
    print("\n======================================")
    print("Đang train model:", model_name)
    print("======================================")

    # Train bằng clean_data
    model.fit(X_train, y_train)
    trained_models[model_name] = model

    # Dự đoán trên clean_test_data
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Đánh giá
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

# ===============================
# 8. BẢNG SO SÁNH MODEL
# ===============================

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="F1-score", ascending=False)

print("\n=== BẢNG SO SÁNH 4 MODEL TRÊN FILE TEST ===")
display(results_df)

# Lưu bảng kết quả
results_output_path = "/content/model_comparison_results.csv"
results_df.to_csv(results_output_path, index=False)

print("Đã lưu bảng so sánh model tại:", results_output_path)

# ===============================
# 9. CHỌN MODEL TỐT NHẤT
# ===============================

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\nModel tốt nhất theo F1-score:", best_model_name)

# ===============================
# 10. XUẤT DỰ ĐOÁN CỦA MODEL TỐT NHẤT
# ===============================

best_pred = best_model.predict(X_test)
best_proba = best_model.predict_proba(X_test)[:, 1]

best_submission = pd.DataFrame({
    "CustomerID": customer_id,
    "Actual_Churn": y_test,
    "Predicted_Churn": best_pred,
    "Churn_Probability": best_proba
})

best_output_path = "/content/churn_prediction_best_model.csv"
best_submission.to_csv(best_output_path, index=False)

print("\nĐã lưu file dự đoán model tốt nhất tại:", best_output_path)
display(best_submission.head())